In [ ]:
from config_dirs import CLEANING_INPUT, FINAL_OUTPUT

import sys
import pandas as pd
# To display all rows 
pd.set_option('display.max_rows', None)
# To display all columns
pd.set_option('display.max_columns', None)
# To prevent truncation of wide columns (full content of each column)
pd.set_option('display.max_colwidth', None)

In [ ]:
# Load the Excel file into a DataFrame
gwas_to_clean_df = pd.read_csv(CLEANING_INPUT)

gwas_to_clean_df['PMID'].nunique()

# Extract unique PMIDs from the 'PMID' column and create a new dataframe
GWAS_unique_PMIDs = pd.DataFrame(gwas_to_clean_df['PMID'].unique(), columns=['PMID'])

# Display the first few rows of the new dataframe
GWAS_unique_PMIDs.head()

In [ ]:
GWAS_unique_PMIDs.shape

In [ ]:
import pandas as pd
import time
import os
from Bio import Entrez

# Set your email and (optionally) your API key for Entrez
Entrez.email = "your_email@example.com"
# Entrez.api_key = "YOUR_NCBI_API_KEY"  # Uncomment and set if you have an API key

# Name of the file where results will be saved incrementally
output_filename = "pubmed_results.csv"

if os.path.exists(output_filename):
    print("Found an existing results file. Loading it to resume progress...")
    partial_df = pd.read_csv(output_filename)
    # Ensure the PMIDs are stored as strings for consistency.
    processed_pmids = set(partial_df['PMID'].astype(str))
    # Convert MeSH terms back from string representation if needed.
    results = partial_df.to_dict('records')
else:
    processed_pmids = set()
    results = []

# Total number of PMIDs to process
total_pmids = len(GWAS_unique_PMIDs)

# Iterate over each PMID from your unique PMIDs dataframe
for idx, row in GWAS_unique_PMIDs.iterrows():
    pmid = str(row['PMID'])
    
    # Skip PMIDs that have already been processed
    if pmid in processed_pmids:
        print(f"PMID {pmid} already processed. Skipping.")
        continue

    print(f"Processing PMID {pmid} ({idx + 1} of {total_pmids})")
    
    try:
        # Fetch the record from PubMed in XML format
        with Entrez.efetch(db="pubmed", id=pmid, retmode="xml") as handle:
            records = Entrez.read(handle)
        
        # Check that we got at least one article
        if records.get('PubmedArticle'):
            article = records['PubmedArticle'][0]
            medline = article.get('MedlineCitation', {})
            article_info = medline.get('Article', {})
            
            # --- Extract the Abstract ---
            abstract_text = None
            if "Abstract" in article_info and "AbstractText" in article_info["Abstract"]:
                abs_parts = article_info["Abstract"]["AbstractText"]
                # abs_parts may be a list or a single string. Combine if needed.
                if isinstance(abs_parts, list):
                    abstract_text = " ".join([str(part) for part in abs_parts])
                else:
                    abstract_text = str(abs_parts)
            
            # --- Extract the MeSH Terms ---
            mesh_terms_list = None
            if "MeshHeadingList" in medline:
                mesh_terms_list = []
                for mesh in medline["MeshHeadingList"]:
                    # Each mesh heading typically contains a DescriptorName element.
                    if "DescriptorName" in mesh:
                        mesh_terms_list.append(str(mesh["DescriptorName"]))
            # (If there are no MeshHeadingList entries, mesh_terms_list remains None.)
            
            # Append the result for this PMID
            results.append({
                "PMID": pmid,
                "Abstract_body": abstract_text,
                "MeshTerms": mesh_terms_list
            })
        else:
            # If no article was returned for this PMID
            results.append({
                "PMID": pmid,
                "Abstract_body": None,
                "MeshTerms": None
            })
        # Mark this PMID as processed
        processed_pmids.add(pmid)
    
    except Exception as e:
        print(f"Error processing PMID {pmid}: {e}")
        # In case of an error, wait a bit before continuing
        time.sleep(5)
        continue  # Skip to the next PMID
    
    # Sleep to avoid hitting the rate limit.
    # (NCBI recommends no more than 3 requests per second.)
    time.sleep(0.4)
    
    # Save intermediate results every 10 PMIDs (you can adjust this frequency)
    if (idx + 1) % 10 == 0:
        temp_df = pd.DataFrame(results)
        temp_df.to_csv(output_filename, index=False)
        print(f"Intermediate results saved after processing {idx + 1} records.")

# Save the final results once all PMIDs have been processed.
final_df = pd.DataFrame(results)
final_df.to_csv(FINAL_OUTPUT, index=False)
print(f"Finished processing all PMIDs. Final results saved to '{output_filename}'.")


In [ ]:
import pandas as pd

# Load the full output CSV
final_df = pd.read_csv(output_filename)

# -------------------------------
# 1. Filter for cardiovascular-related entries
# - Keep rows where 'Causality' contains "Causal"
# - Keep rows where 'Organ/System' contains 'Cardiovascular' (case-insensitive) or 'vascular'
# -------------------------------
cardio_df = final_df[
    final_df['Causality'].str.contains('Causal', case=False, na=False) &
    final_df['Organ/System'].str.contains('Cardiovascular|vascular', case=False, na=False)
]
# Save cardiovascular filtered data
cardio_df.to_csv('cardiovascular.csv', index=False)
print("Cardiovascular filtered data saved to 'cardiovascular.csv'")

# -------------------------------
# 2. Filter for brain-related entries
# - Keep rows where 'Causality' contains "Causal"
# - Keep rows where 'Organ/System' contains 'Brain', 'nervous system' or 'cerebrovascular' (case-insensitive)
# -------------------------------
brain_df = final_df[
    final_df['Causality'].str.contains('Causal', case=False, na=False) &
    final_df['Organ/System'].str.contains('Brain|nervous system|cerebrovascular', case=False, na=False)
]
# Save brain filtered data
brain_df.to_csv('brain.csv', index=False)
print("Brain filtered data saved to 'brain.csv'")

# -------------------------------
# 3. Find shared genes between cardiovascular and brain
# - Using the 'Perplexity' column to identify common genes
# - Keep all other columns from each dataset
# -------------------------------
shared_genes_df = pd.merge(
    cardio_df, 
    brain_df, 
    on='Perplexity',  # Intersection based on Perplexity (gene name)
    suffixes=('_cardio', '_brain')
)

# Save the intersection to Excel
shared_genes_df.to_excel('Shared_genes_final.xlsx', index=False)
print("Shared genes saved to 'Shared_genes_final.xlsx'")
